# SkinIQ — training experiments (multi-label skin concern model)

Three experiments on the EfficientNet-V2-S multi-label model. Each changes **one** training setting;
everything else stays at the **baseline**, v2's settings: strong augmentation, learning rate 2e-4,
dropout 0.3, 3 frozen-backbone epochs, up to 12 epochs, batch size 32, image size 224. All runs use the
same data and **your relabelled labels** (merged from Drive), so the comparison is fair.

| Experiment | Setting changed | Values compared (baseline in bold) |
|---|---|---|
| 1 | Data augmentation | mild, **strong** |
| 2 | Learning rate | 1e-4, **2e-4**, 3e-4 |
| 3 | Dropout rate | 0.2, **0.3**, 0.4 |

**Baseline run:** `skin_concerns_v4` already is v2's settings trained on your labels, so it is reused
(set `RETRAIN_BASELINE = True` below if you changed labels since v4). That leaves **5 new runs, about
20 min each** on a T4 GPU. You can run one experiment per Colab session: finished runs are skipped.

**Choosing the best value:** by the **validation** mean ROC-AUC (best epoch), never by the test results;
test results are reported for each run. Results go to `My Drive/SkinIQ/models/experiments/`.

**Run:** Runtime → T4 GPU, run sections 0–2, then the experiment cells you want, then section 4.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/SkinIQ"
SKIN_TYPE_ZIP = "Skin Type Dataset - olimov.zip"
SKIN_CONDITION_ZIP = "Skin Issues Dataset - drishya.zip"
SKIN_PROBLEM_ZIP = "Skin Problem Detection.zip"   # Roboflow, CC BY 4.0
EPOCHS = 12
BATCH_SIZE = 32          # lower to 16 if you get "CUDA out of memory" (then retrain the baseline too)
EXP_DIR = f"{DRIVE_DIR}/models/experiments"
BASELINE_DIR = f"{DRIVE_DIR}/models/skin_concerns_v4"   # v2 settings on your labels
RETRAIN_BASELINE = False

## 1. Set up (about 5 minutes)

In [ ]:
import torch, os
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again"
from google.colab import drive
drive.mount("/content/drive")
LABELS = f"{DRIVE_DIR}/labels/multilabel_labels.csv"
for f in (SKIN_TYPE_ZIP, SKIN_CONDITION_ZIP, SKIN_PROBLEM_ZIP):
    assert os.path.exists(f"{DRIVE_DIR}/{f}"), f"Not found: {DRIVE_DIR}/{f}"

%cd /content
!rm -rf SkinIQ && git clone -q https://github.com/rwiie/SkinIQ.git
%cd /content/SkinIQ
!pip install -q "timm>=1.0" opencv-python-headless
!mkdir -p data/raw
!cp "{DRIVE_DIR}/{SKIN_TYPE_ZIP}" "{DRIVE_DIR}/{SKIN_CONDITION_ZIP}" /tmp/
!unzip -q -o "/tmp/{SKIN_TYPE_ZIP}" -x "__MACOSX/*" -d data/raw
!unzip -q -o "/tmp/{SKIN_CONDITION_ZIP}" -x "__MACOSX/*" -d data/raw
!rm -rf data/raw/skin_type data/raw/skin_condition
!mv "data/raw/Skin Type Dataset - olimov" data/raw/skin_type
!mv "data/raw/Skin Issues Dataset - drishya" data/raw/skin_condition
!python training/prepare_data.py
!rm -rf data/raw/skin_problem && mkdir -p data/raw/skin_problem
!cp "{DRIVE_DIR}/{SKIN_PROBLEM_ZIP}" /tmp/
!unzip -q -o "/tmp/{SKIN_PROBLEM_ZIP}" -d data/raw/skin_problem
!python training/prepare_roboflow.py
# Labels: the committed test/validation labels (data/labels/multilabel_labels.csv), plus any
# you added in Drive with annotate_colab.ipynb (Drive wins where the same photo is in both).
import pandas as pd
repo_labels = pd.read_csv("data/labels/multilabel_labels.csv")
if os.path.exists(LABELS):
    drive_labels = pd.read_csv(LABELS)
    repo_labels = pd.concat([repo_labels[~repo_labels.key.isin(drive_labels.key)], drive_labels])
    print(f"Added {len(drive_labels)} labels from Drive")
repo_labels.to_csv("data/labels/multilabel_labels.csv", index=False)
print(f"{len(repo_labels)} labelled photos")

## 2. Label summary

In [ ]:
import pandas as pd
labels = pd.read_csv("data/labels/multilabel_labels.csv")
usable = labels[labels.unusable == 0]
concerns = ["acne", "blackheads", "hyperpigmentation", "redness", "enlarged_pores", "wrinkles"]
summary = usable.groupby("split")[concerns].sum()
summary["clear"] = usable[usable[concerns].sum(axis=1) == 0].groupby("split").size()
summary["2+ concerns"] = usable[usable[concerns].sum(axis=1) >= 2].groupby("split").size()
summary["photos"] = usable.groupby("split").size()
display(summary.fillna(0).astype(int))
n_test, n_val = (usable.split == "test").sum(), (usable.split == "val").sum()
print(f"test {n_test} / 300, val {n_val} / 200 labelled")
if n_test < 300 or n_val < 200:
    print("⚠️  Fewer than the recommended test/val labels: results will be less reliable.")

## 3. Runs

`run()` trains one setting and saves it to `EXP_DIR/<name>/`; a run that already has results is skipped.

In [ ]:
import os, shutil
BASE = {"aug": "strong", "lr": 2e-4, "drop_rate": 0.3, "freeze_epochs": 3}
EXPERIMENTS = {
    "1. Data augmentation": [("baseline", {}), ("exp1_aug_mild", {"aug": "mild"})],
    "2. Learning rate": [("exp2_lr_1e-4", {"lr": 1e-4}), ("baseline", {}), ("exp2_lr_3e-4", {"lr": 3e-4})],
    "3. Dropout rate": [("exp3_dropout_0.2", {"drop_rate": 0.2}), ("baseline", {}), ("exp3_dropout_0.4", {"drop_rate": 0.4})],
}
os.makedirs(EXP_DIR, exist_ok=True)

def run(name, change):
    out = f"{EXP_DIR}/{name}"
    if os.path.exists(f"{out}/metrics.json"):
        print(f"✓ {name}: already done"); return
    if name == "baseline" and not RETRAIN_BASELINE and os.path.exists(f"{BASELINE_DIR}/metrics.json"):
        shutil.copytree(BASELINE_DIR, out, dirs_exist_ok=True)
        print(f"✓ baseline: copied from {BASELINE_DIR}"); return
    s = {**BASE, **change}
    print(f"▶ {name}: {s}")
    get_ipython().system(f'python training/train_multilabel.py --epochs {EPOCHS} --batch-size {BATCH_SIZE} '
                         f'--out-dir "{EXP_DIR}" --run-name {name} --aug {s["aug"]} --lr {s["lr"]} '
                         f'--drop-rate {s["drop_rate"]} --freeze-epochs {s["freeze_epochs"]}')
    assert os.path.exists(f"{out}/metrics.json"), f"{name} failed: scroll up for the error and send it to Claude"

run("baseline", {})

### Experiment 1: data augmentation (1 new run, ~20 min)

In [ ]:
for name, change in EXPERIMENTS["1. Data augmentation"]:
    run(name, change)

### Experiment 2: learning rate (2 new runs, ~40 min)

In [ ]:
for name, change in EXPERIMENTS["2. Learning rate"]:
    run(name, change)

### Experiment 3: dropout rate (2 new runs, ~40 min)

In [ ]:
for name, change in EXPERIMENTS["3. Dropout rate"]:
    run(name, change)

## 4. Results (run any time)

One table per experiment. **Selected** = highest validation mean ROC-AUC. Test columns are for reporting
only. Saved as `EXP_DIR/experiment_results.csv`.

In [ ]:
import json
import pandas as pd

def summarise(name):
    d = f"{EXP_DIR}/{name}"
    if not os.path.exists(f"{d}/metrics.json"):
        return None
    m = json.load(open(f"{d}/metrics.json"))
    h = pd.read_csv(f"{d}/history.csv")
    per = pd.read_csv(f"{d}/per_concern.csv").set_index("concern")
    s = m["settings"]
    row = {"run": name, "augmentation": s["aug"], "learning rate": float(s["lr"]), "dropout": float(s["drop_rate"]),
           "best epoch": m["best_epoch"], "val mean ROC-AUC": h["val_mean_roc_auc"].max(),
           "test macro F1": m["test"]["macro_f1"], "test micro F1": m["test"]["micro_f1"],
           "test macro precision": round(per.precision.mean(), 4), "test macro recall": round(per.recall.mean(), 4),
           "test mean ROC-AUC": round(per.roc_auc.mean(), 4), "exact match": m["test"]["exact_match"],
           "hamming loss": m["test"]["hamming_loss"], "clear-skin false alarms": m["test"]["clear_false_alarm_rate"]}
    row.update({f"F1 {c}": per.loc[c, "f1"] for c in per.index})
    return row

tables = []
for exp, runs in EXPERIMENTS.items():
    rows = [r for r in (summarise(n) for n, _ in runs) if r]
    if not rows:
        print(f"{exp}: no results yet"); continue
    t = pd.DataFrame(rows)
    t.insert(0, "experiment", exp)
    t["selected"] = (t["val mean ROC-AUC"] == t["val mean ROC-AUC"].max()).map({True: "✓", False: ""})
    print(f"\n{exp}")
    display(t.drop(columns="experiment").set_index("run").round(4))
    tables.append(t)
if tables:
    pd.concat(tables).to_csv(f"{EXP_DIR}/experiment_results.csv", index=False)
    print(f"Saved {EXP_DIR}/experiment_results.csv: share the experiments folder link with Claude.")

## 5. Same photos as v2, your labels (1 run, ~20 min)

v2 and v4 differ in two ways: who labelled (Claude vs you) **and** how many labelled training photos
(265 vs 439). This run changes **only the labels**: v2's settings and v2's exact photos (265 training,
200 validation, 237 test), all with **your** labels (`data/labels/v2_photos_author_labels.csv`; 3 photos you
marked unusable are left out, your 1 unsure test photo is kept so the test set stays v2's 237).

Then v2, this run and v4 are all scored on those same 237 test photos with your labels, thresholds re-tuned on
the same validation photos. Run section 1 first (data set-up). Results: `My Drive/SkinIQ/models/v2_photos_author_labels/`.


In [ ]:
SAME_LABELS = "data/labels/v2_photos_author_labels.csv"
RUN = "skin_concerns_v2_author"
OUT = f"{DRIVE_DIR}/models"
if os.path.exists(f"{OUT}/{RUN}/metrics.json"):
    print(f"✓ {RUN}: already done")
else:
    get_ipython().system(f'python training/train_multilabel.py --labels {SAME_LABELS} --epochs 12 --batch-size {BATCH_SIZE} '
                         f'--out-dir "{OUT}" --run-name {RUN} --aug strong --lr 2e-4 --drop-rate 0.3 '
                         f'--freeze-epochs 3 --include-unsure-in-test')
    assert os.path.exists(f"{OUT}/{RUN}/metrics.json"), f"{RUN} failed: scroll up for the error and send it to Claude"


In [ ]:
# Score v2, this run and v4 on the same 237 test photos (your labels)
from skiniq.vision.weights import ensure_model
v2 = ensure_model()   # the app's v2 weights (downloaded and checked)
models = f'--model v2={v2} --model v2_your_labels="{OUT}/{RUN}/model.pt"'
if os.path.exists(f"{BASELINE_DIR}/model.pt"):
    models += f' --model v4="{BASELINE_DIR}/model.pt"'
CMP = f"{DRIVE_DIR}/models/v2_photos_author_labels"
get_ipython().system(f'python training/evaluate_multilabel.py --labels {SAME_LABELS} {models} '
                     f'--include-unsure-in-test --out-dir "{CMP}"')
cmp = pd.read_csv(f"{CMP}/comparison.csv")
display(cmp[cmp.thresholds == "retuned on these val labels"].drop(columns="threshold_values").set_index("model").round(4))
print(f"Saved {CMP}: share that folder link with Claude.")
